# Project #2 Short-Answer Questions

ICS 3201 Computer Security and Cryptography — Double Ratchet Messenge

## Question 1

**In our implementation, Alice and Bob increment their Diffie-Hellman ratchets every time they exchange messages. Could the protocol be modified to have them increment the DH ratchets once every ten messages without compromising confidentiality against an eavesdropper (i.e., semantic security)?**

Yes. Semantic security (IND-CPA) against a purely **passive eavesdropper** comes from the **symmetric-key (chain) ratchet**, not the DH ratchet: every message key is produced by running the current chain key through an HMAC (a PRF) with a fixed public constant, so each message key is still computationally indistinguishable from random no matter how many messages separate two DH ratchet steps.

An eavesdropper who never learns any secret state gains nothing from DH ratchets happening less frequently, since she never has a starting point (a leaked key) from which to exploit the lack of fresh DH randomness. The DH ratchet exists to provide **forward secrecy and break-in recovery under active key compromise**, not confidentiality against passive observation — so reducing its frequency to once every ten messages would not, by itself, break semantic security.

## Question 2

**What if they never update their DH keys at all? Please explain the security consequences of this change with regards to Forward Secrecy and Break-in Recovery.**

If the DH ratchet never runs, the entire session relies on a single DH-derived root key forever, with only the symmetric chain ratchet (HMAC) advancing over time.

- **Forward Secrecy — partially preserved.** The symmetric chain ratchet is one-way (HMAC cannot be inverted), so even with the current chain key in hand, an attacker cannot recompute *earlier* chain keys or message keys. Messages sent strictly before a compromise remain safe.
- **Break-in Recovery — completely lost.** Without a DH ratchet, no fresh secret randomness is ever injected into the session after it starts. Once an attacker learns the current chain key (and the static DH secret that is the only thing that ever fed the root key), she can walk the same deterministic HMAC chain forward exactly as the legitimate party would, deriving every future message key as well. There is no mechanism by which a single successfully-delivered message can "heal" the session, because nothing new and secret is ever introduced — compromise becomes permanent rather than recoverable.

## Question 3

**Conversation:**
```
A: Hey Bob, can you send me the locker combo?
A: I need to get my laptop
B: Sure, it's 1234!
A: Great, thanks! I used it and deleted the previous message.
B: Did it work?
```
**What is the length of the longest sending chain used by Alice? By Bob? Please explain.**

Message order by sender: **A, A, B, A, B**.

A DH ratchet step (and therefore a brand-new sending chain) is triggered whenever a party sends a message *after* having received something new since their last send. So a sending chain's length equals the number of messages a party sends consecutively, with no received message in between.

- **Alice:** Chain 1 = {"Hey Bob, can you send me the locker combo?", "I need to get my laptop"} → length **2** (sent back-to-back, no reply from Bob in between). Chain 2 = {"Great, thanks!..."} → length 1 (sent only after receiving Bob's reply, which forces a fresh chain).
  - **Longest sending chain for Alice = 2.**
- **Bob:** Chain 1 = {"Sure, it's 1234!"} → length 1. Chain 2 = {"Did it work?"} → length 1. Every one of Bob's messages replies to something he just received, so each starts its own fresh one-message chain.
  - **Longest sending chain for Bob = 1.**

## Question 4

**Mallory compromises Alice's phone just before her third message. She is unable to determine the locker combination. State and describe the relevant security property and justify why Double Ratchet provides this property.**

**Property: Forward Secrecy.**

The locker combo ("1234") was contained in Bob's message, which Alice already received and decrypted *before* Mallory's compromise. Decrypting that message consumed a receiving-chain key that the Double Ratchet immediately advanced past (via the one-way HMAC chain ratchet) the moment the message was processed — the old chain key and message key were discarded rather than retained (our implementation explicitly overwrites `conn.receivingChainKey`, keeping only O(1) state, no history).

Mallory's snapshot of "Alice's current keys" is therefore *downstream* of that one-way function. Since HMAC cannot be inverted, there is no computational path from Alice's current secret state back to the specific receiving-chain key that decrypted Bob's "1234" message. Mallory gains full ability to decrypt *future* messages (break-in), but cannot recover anything from before the compromise — which is exactly what Forward Secrecy guarantees.

## Question 5

**The method of government surveillance is deeply flawed. Why might it not be as effective as intended? What are the major risks involved?**

- **Single point of catastrophic failure.** Every message from every user is encrypted under one fixed, static government public key. A single leak, breach, insider misuse, or legal compulsion exposing the government's one secret key would retroactively decrypt *all* recorded traffic from *every* user ever sent under that key — completely undoing the forward secrecy and break-in recovery the rest of the protocol provides, for the entire population at once, forever. This mirrors the classic objection to key-escrow schemes like the NSA's Clipper Chip.
- **Trivially bypassable by the people it targets.** Nothing in the protocol cryptographically *proves* that `cGov` correctly encrypts the real message key — there is no zero-knowledge proof or similar binding mechanism. A custom or modified client (exactly the kind a determined bad actor would use) can send garbage, encrypt under the wrong key, or omit the government fields entirely, and two such clients will still communicate perfectly fine with each other. The surveillance burden falls entirely on compliant, law-abiding users while the actual surveillance targets opt out trivially.
- **High-value centralized target.** A single secret key capable of decrypting a whole population's communications is an extraordinarily attractive target for foreign intelligence services, organized crime, or rogue insiders — raising systemic risk for everyone, including people who were never under suspicion.
- **Reintroduces a long-term secret into a forward-secure system.** The entire point of Double Ratchet is that no single long-term key compromise should expose past communications. Bolting on a static escrow key reintroduces precisely that failure mode by design.

## Question 6

**Compare ECDSA vs RSA-based signatures for: (a) key generation time, (b) signing time, (c) signature length, (d) verification time.**

Measured using `question6code/q6code.js` (ECDSA P-384, RSA-PSS 4096-bit modulus, e = 65537), averaged across several runs on Node 18 / SubtleCrypto:

| | RSA (4096-bit) | ECDSA (P-384) | Winner |
|---|---|---|---|
| (a) Key generation | ~175–540 ms | ~1.4–2.6 ms | **ECDSA**, by ~2 orders of magnitude |
| (b) Signing | ~7–9.5 ms | ~1.3–2.0 ms | **ECDSA**, ~4–6x faster |
| (c) Signature length | 512 bytes | 96 bytes | **ECDSA**, >5x shorter |
| (d) Verification | ~0.3–0.44 ms | ~0.9–1.2 ms | **RSA**, ~2–3x faster |

Sample raw output from `node q6code.js`:
```
RSA Key Generation Total Time: 538.442ms
RSA Signing Total Time: 9.506ms
RSA Verify Total Time: 0.44ms
RSA Signature Byte Length: 512

ECDSA Key Generation Total Time: 2.409ms
ECDSA Signing Total Time: 1.96ms
ECDSA Verify Total Time: 1.091ms
ECDSA Signature Byte Length: 96
```

**Why:** RSA key generation requires finding two large probable primes (expensive, variable-time primality testing), and RSA signing is a modular exponentiation with a large private exponent `d` — both inherently costly for a 4096-bit modulus. RSA's signature size is fixed at the modulus size (512 bytes = 4096 bits), far larger than ECDSA's two 48-byte curve-order integers (r, s) for P-384.

RSA's one advantage is verification speed: with a small public exponent (e = 65537 = 0x10001), RSA verification is a cheap modular exponentiation, cheaper than ECDSA's elliptic-curve point-multiplication-based verify. This is why RSA remains attractive in scenarios like certificate chains, where a signature is created once but verified many times.